# YOLOv8-OBB Training: Bottle vs Can Detection (v4 — Real World Data)

**Dataset:** Roboflow v7 (studio) + 104 real webcam images — balanced train/val/test  
**Model:** YOLOv8n-OBB  |  **GPU:** T4  |  **Epochs:** 100

## 1. Environment Setup

In [ ]:
!nvidia-smi

In [ ]:
%pip install -q ultralytics
import ultralytics
ultralytics.checks()

## 2. Load Dataset from Google Drive

In [ ]:
from google.colab import drive
import shutil, os
from pathlib import Path

drive.mount("/content/drive")

ZIP_PATH = "/content/drive/MyDrive/dataset_v4.zip"
assert os.path.exists(ZIP_PATH), f"Not found: {ZIP_PATH} — upload dataset_v4.zip to Drive root"

!unzip -q "{ZIP_PATH}" -d /content/

# Roboflow unzips into data/v8_balanced — move to /content/dataset
src = Path("/content/data/v8_balanced")
dst = Path("/content/dataset")
if dst.exists():
    shutil.rmtree(dst)
shutil.move(str(src), str(dst))

# Fix data.yaml paths for Colab
dst.joinpath("data.yaml").write_text(
    "path: /content/dataset\n"
    "train: train/images\n"
    "val: valid/images\n"
    "test: test/images\n\n"
    "names:\n"
    "  0: bottle\n"
    "  1: can\n\n"
    "nc: 2\n"
)

for split in ["train", "valid", "test"]:
    n = len(list((dst / split / "images").glob("*.jpg")))
    print(f"{split:5s}: {n} images")

## 3. Class Balance Check

In [ ]:
from collections import Counter

for split in ["train", "valid", "test"]:
    c = Counter()
    for lbl in (dst / split / "labels").glob("*.txt"):
        for line in lbl.read_text().strip().splitlines():
            if line:
                c[int(line.split()[0])] += 1
    t = sum(c.values())
    print(f"{split:5s}: bottle={c[0]:4d} ({100*c[0]//max(t,1)}%)  "
          f"can={c[1]:4d} ({100*c[1]//max(t,1)}%)  total={t}")

## 4. Training

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n-obb.pt")

results = model.train(
    data="/content/dataset/data.yaml",
    epochs=100,
    imgsz=640,
    batch=16,
    optimizer="AdamW",
    lr0=0.001,
    cos_lr=True,
    patience=20,
    mosaic=1.0,
    mixup=0.15,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=15.0,
    translate=0.1,
    scale=0.5,
    project="/content/runs",
    name="bottle_can_obb_v4",
    exist_ok=True,
    plots=True,
)

## 5. Evaluate on Test Set

In [ ]:
best = "/content/runs/bottle_can_obb_v4/weights/best.pt"
model = YOLO(best)
metrics = model.val(data="/content/dataset/data.yaml", split="test", plots=True)

print(f"mAP@0.5      : {metrics.box.map50:.4f}")
print(f"mAP@0.5:0.95 : {metrics.box.map:.4f}")
per_class = dict(zip(["bottle", "can"], metrics.box.maps))
print(f"bottle mAP@0.5: {per_class['bottle']:.4f}")
print(f"can    mAP@0.5: {per_class['can']:.4f}")

## 6. Save to Google Drive

In [ ]:
DRIVE_PROJECT = "/content/drive/MyDrive/yolo-portfolio"
os.makedirs(f"{DRIVE_PROJECT}/models", exist_ok=True)

shutil.copy(best, f"{DRIVE_PROJECT}/models/best.pt")
print("Weights saved:", f"{DRIVE_PROJECT}/models/best.pt")

if os.path.exists(f"{DRIVE_PROJECT}/results/training"):
    shutil.rmtree(f"{DRIVE_PROJECT}/results/training")
shutil.copytree("/content/runs/bottle_can_obb_v4", f"{DRIVE_PROJECT}/results/training")
print("Training results saved to Drive.")

## 7. Download best.pt directly

In [ ]:
from google.colab import files
files.download(best)